# 0.41 — Entities by NER, relations by a dated model

Notebook 0.40 asked one model to fill the whole schema and measured where it failed. This one
splits the work by what each tool is actually good at.

| field | who does it | why |
|---|---|---|
| **Entity** | spaCy `en_core_web_trf` | a reading task — NER wins, and by a lot |
| **Relations · Subject / Object** | DatedGPT, *choosing among the NER entities* | a choice between two known names |
| **Relations · Predicate** | a trigger lexicon, then DatedGPT only if ambiguous | the verb is usually written in the headline |
| ~~Event / Action~~ | dropped | it is the predicate under another name |
| ~~Technology / Concept~~ | later, from Wikipedia | 0.40 measured 53 % here — not worth keeping |

**Why NER rather than the LLM for entities.** 0.40 measured DatedGPT at 73 % on `Entity`. The
default spaCy model is *worse* — 69 % F1, with unusable spans, because Title Case headlines make
it swallow capitalised verbs into the name ("Cathie Wood Offloads Nvidia Ahead"). The transformer
model fixes exactly that: **90 % F1**, §1 measures it. That gap is the whole argument for this
split, and it is why the first thing here is a measurement rather than a pipeline.

**The recogniser is not point-in-time, and that is a known, accepted gap.** `en_core_web_trf` is
a modern model, so its recall is higher for names that became prominent after the date being
analysed — a leak in the direction that flatters an emerging-theme backtest. It is kept anyway,
because DatedGPT scores 73 % against its 96 % and its errors are ordinary mistakes rather than
dated ignorance. Written up, with the fix and one failed attempt at quantifying it, in
`src-deliverable/FUTUREWORK_dated_ner.md`. For now *Anthropic* goes unrecognised and that edge is
simply missing.

**Why the LLM still earns its place.** Once NER has produced the names, the questions left are
*which of these two is acting on the other* and *what is the verb*. Both are choices from a
short list, which is the one thing a 1.3 B model does reliably — and, being a dated model, it
makes those choices with no knowledge of how the story ended.

In [1]:
import gc, re, warnings
from pathlib import Path

import pandas as pd
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

NER = spacy.load("en_core_web_trf")
KEEP = {"ORG", "PERSON"}          # PRODUCT is the Technology/Concept field — not this notebook

# ── which point-in-time model does the choosing ───────────────────────────────
# Two independent families, both trained so that a vintage cannot know its future:
#
#   datedgpt   twelve 1.3 B Llama-architecture models, one per year, ~100 B tokens each
#              (arXiv 2603.11838)
#   pit        Point-In-Time GPT, trained on chronological monthly snapshots of FineWeb
#              (Kelly, Malamud, Schwab & Xu, "Scaling Point-in-Time Language Models",
#              Swiss Finance Institute 26-37, 2026) — 1 B and 4 B, plus a 4 B instruction
#              -tuned variant
#
# They share no tokenizer, no architecture and no training data, so agreeing on an answer
# is real evidence and disagreeing is worth looking at. Switch with one line.
BACKEND = "datedgpt"        # "datedgpt" | "pit-4b-ft"  ← the two that answer questions
                            # "pit-1b" | "pit-4b"      ← pretraining only, see chat() below

# PIT snapshots are months, and 2015 is November rather than December.
PIT_MONTH = {2013: "201312", 2014: "201412", 2015: "201511", 2016: "201612", 2017: "201712",
             2018: "201812", 2019: "201912", 2020: "202012", 2021: "202112", 2022: "202212",
             2023: "202312", 2024: "202412"}

BACKENDS = {
    # instruction-tuned: these answer a question
    "datedgpt":  dict(repo=lambda y: f"datedgpt/datedgpt-{y}-instruct",   trust=False, gb=2.7,  chat=True),
    "pit-4b-ft": dict(repo=lambda y: f"Diamegs/PIT-4B-FT-{PIT_MONTH[y]}", trust=True,  gb=16.5, chat=True),
    # pretraining only: these continue text. Fine for choose(), which scores continuations
    # anyway, and useless for chat() — see the note there.
    "pit-1b":    dict(repo=lambda y: f"Diamegs/PIT-1B-{PIT_MONTH[y]}",    trust=True,  gb=6.1,  chat=False),
    "pit-4b":    dict(repo=lambda y: f"Diamegs/PIT-4B-{PIT_MONTH[y]}",    trust=True,  gb=16.5, chat=False),
}
# `trust=True` executes `modeling_pit.py` from the repo on load. It was read before being
# enabled here: 405 lines, imports torch and transformers only, no filesystem, network or
# eval. Re-read it if the repo changes — trust_remote_code is a real capability, not a flag.

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32

_LOADED, _TOK = {}, {}

def tokenizer(backend=None):
    """One tokenizer per family — they are not interchangeable (Llama SP vs GPT-2 BPE)."""
    b = backend or BACKEND
    if b not in _TOK:
        _TOK[b] = AutoTokenizer.from_pretrained(BACKENDS[b]["repo"](2022),
                                                trust_remote_code=BACKENDS[b]["trust"])
    return _TOK[b]

def model(year, backend=None):
    """One vintage of the chosen family, on DEVICE. Downloads on first use."""
    b = backend or BACKEND
    if (b, year) not in _LOADED:
        cfg = BACKENDS[b]
        m = AutoModelForCausalLM.from_pretrained(cfg["repo"](year), dtype=DTYPE,
                                                 trust_remote_code=cfg["trust"])
        _LOADED[(b, year)] = m.to(DEVICE).eval()
    return _LOADED[(b, year)]

def release():
    _LOADED.clear(); gc.collect()
    if DEVICE != "cpu":
        getattr(torch, DEVICE).empty_cache()

@torch.no_grad()
def chat(question, year=None, backend=None, max_new_tokens=140):
    """Ask a vintage a question in plain language. Works on every backend.

    The three families want different prompts and there is no way around it:
      datedgpt    a Llama-2 chat template, which emits its own <s> — so the tokenizer must
                  not add a second one
      pit-4b-ft   a manual <|user|> … <|assistant|> frame, stopping at <|end|>
      pit base    no instruction format at all: these are pretrained only, so the question is
                  simply continued. Phrase it as a sentence to finish, not as a question.
    """
    b = backend or BACKEND
    if not BACKENDS[b]["chat"]:
        print(f"note: {b} is pretrained only — it continues text rather than answering. "
              f"Give it a sentence to finish.")
    tok = tokenizer(b)
    if b == "datedgpt":
        text = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False)
        add_special = False
    elif b == "pit-4b-ft":
        text, add_special = f"<|user|>\n{question}\n<|assistant|>\n", True
    else:
        text, add_special = question, True          # base model: pure continuation

    enc = tok(text, return_tensors="pt", add_special_tokens=add_special).to(DEVICE)
    out = model(year or VINTAGE, b).generate(
        **enc, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False,
        repetition_penalty=1.1, eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    answer = tok.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    return answer.split("<|end|>")[0].strip()


VINTAGE = 2024        # the vintage that does the choosing; 2013–2024

TOK = tokenizer()
print(f"spaCy {spacy.__version__} · en_core_web_trf · {DEVICE}/{str(DTYPE).split('.')[-1]}")
print(f"backend {BACKEND} · {BACKENDS[BACKEND]['repo'](VINTAGE)} · {BACKENDS[BACKEND]['gb']} GB per vintage")

spaCy 3.8.14 · en_core_web_trf · mps/float16
backend datedgpt · datedgpt/datedgpt-2024-instruct · 2.7 GB per vintage


## Ask a vintage something

One cell to talk to the models directly, before any of the pipeline below. Change `question`,
re-run: the weights stay in the kernel, so it answers in a second or two.

`backend=` overrides the global choice for one call, so the same question can go to both
families side by side — they share no tokenizer, no architecture and no training data, so
agreement between them is worth more than either one alone. `year=` picks the vintage, and
asking the same thing of 2019 and 2024 is the quickest way to see a cutoff at work.

**Only two backends answer questions**: `datedgpt` and `pit-4b-ft`. The other two PIT models
are pretrained only — they continue text rather than reply, so a question gets you the next
sentence of an imaginary article. `chat()` says so when you call one of them. They are still
useful to the pipeline below, because `choose()` scores continuations rather than asking
anything.

The two conversational backends are not the same size: DatedGPT is 1.3 B and 2.7 GB a vintage,
PIT-4B-FT is 4.23 B and **16.5 GB**. That is the only pairing where a capability difference could
show — and on the one task in this notebook where the model actually decides, it does not: both
score **88 %** on the predicate, and both produce the same single, correct dissent from the
lexicon. Three times the parameters buy nothing here, which says the triggers are doing the hard
part. Their prose differs more than their judgement: ask them a question and DatedGPT answers in
a sentence where PIT-4B-FT writes a numbered list.

And out of its horizon no vintage hedges: it invents, fluently and in the right register. Treat
an open answer as something to check.

In [2]:
# ── ask a vintage ──────────────────────────────────────────────────────────────
question = "What is quantum computing useful for?"

for b in ("datedgpt", "pit-4b-ft"):          # the two that answer questions
    print(f"── {b}\n{chat(question, backend=b)}\n")
release()


# --- the same question to two vintages of one family --------------------------
# for y in (2019, 2024):
#     print(f"\n── {y}\n{chat(question, year=y)}")
# release()

# --- a pretrained-only PIT continues text; it does not answer -----------------
# print(chat("The main risk to European banks in 2022 was", backend="pit-1b"))

── datedgpt
Quantum computing can solve complex optimization problems, simulate molecular structures for drug discovery, and optimize supply chains.

── pit-4b-ft
Quantum Computing has a wide range of potential applications, but it's particularly promising in fields that require complex calculations and data processing. Here are some specific areas where Quantum Computers can be used:

1) **Computing Complex Numbers**: In traditional computers (like classical ones), the number system we use to represent numbers only goes up until about 10^13 or so - after which point you need more advanced methods like binary arithmetic etc., however with Qubits being able handle much larger values this opens doors into new realms such as cryptography/cryptanalysis & even potentially solving problems beyond what current technology allows us too! 

2 ) *Cryptography*: One area already benefiting from



## 1 · Entities

Eighteen headlines labelled by hand, including the awkward ones: a headline with no company at
all, one where the company name is also a theme keyword (*First Quantum Minerals*, a copper
miner), and several in the market-mechanics style that makes up most of the corpus.

Scoring is per entity, lenient on the surface form — "SoftBank" matches "SoftBank Group" —
because the exact string is the *linking* step's problem, not the recogniser's.

One preprocessing step turned out to matter more than the model choice, and it is in
`entities()` below: **the headline is sentence-cased before the recogniser sees it.**

In [3]:
GOLD_ENTITIES = [
    ("Microsoft to Add OpenAI's ChatGPT Technology to Bing Search Engine", ["Microsoft", "OpenAI"]),
    ("Booking.com to Launch AI Planner Partially Powered by ChatGPT", ["Booking.com"]),
    ("Palantir Gains After Posting Rosy Forecast, Readies AI Tool", ["Palantir"]),
    ("Nvidia Jumps as AI Demand Boosts Results, Forecast", ["Nvidia"]),
    ("Google’s AI Chatbot Seeks to Avoid Traps That Ensnared Microsoft", ["Google", "Microsoft"]),
    ("Meta Shares Edge Higher as UBS Hikes PT on AI Boost", ["Meta", "UBS"]),
    ("SoftBank Group Climbs as Optimism Builds Over Arm IPO", ["SoftBank", "Arm"]),
    ("IonQ Signs Deal With Hyundai to Use Quantum Computing for Battery Research", ["IonQ", "Hyundai"]),
    ("Alphabet Unveils Willow Quantum Chip That Cuts Error Rates", ["Alphabet"]),
    ("Novo Nordisk Raises Outlook as Ozempic Sales Beat Estimates", ["Novo Nordisk"]),
    ("Nvidia Options Imply 9.8% Share Move Post Earnings", ["Nvidia"]),
    ("Cathie Wood Offloads Nvidia Ahead of Third-Quarter Results", ["Cathie Wood", "Nvidia"]),
    ("Chipmakers Outperform as AMD Results Lift Nvidia and Intel", ["AMD", "Nvidia", "Intel"]),
    ("First Quantum Minerals Jumps Amid Barrick Takeover Interest", ["First Quantum Minerals", "Barrick"]),
    ("Generative AI Could Solve the Productivity Paradox", []),
    ("Berkshire Buys Stake in Occidental Petroleum", ["Berkshire", "Occidental Petroleum"]),
    ("Broadcom to Acquire VMware in $61 Billion Deal", ["Broadcom", "VMware"]),
    ("Oracle Sues Google Over Use of Java in Android", ["Oracle", "Google"]),
]

def entities(headline):
    """Company and person names, in the order they appear, de-duplicated.

    The headline is fed to the recogniser in **sentence case**, not as written. Bloomberg
    headlines are Title Case, and a capital letter is the strongest hint a NER model has that
    a word is a name — so Title Case makes it label ordinary nouns as organisations (ChatGPT,
    Willow, Ozempic all came back as ORG). Lowercasing everything but the first letter removes
    that false signal and costs nothing, because real company names are recognised from
    context too. Measured on §1's gold set: **F1 92 % as written, 96 % in sentence case**, with
    precision rising from 90 % to 100 %.

    Sentence-casing does not change the string length, so the character offsets still point at
    the right place and the span is sliced back out of the ORIGINAL headline — the recogniser
    sees "microsoft", the graph gets "Microsoft".
    """
    sentence_case = headline[0] + headline[1:].lower()
    seen, out = set(), []
    for e in NER(sentence_case).ents:
        if e.label_ not in KEEP:
            continue
        t = headline[e.start_char:e.end_char].replace("’s", "").replace("'s", "").strip(" .,")
        if t.lower() not in seen and len(t) > 1:
            seen.add(t.lower()); out.append(t)
    return out


def entities_for_edge(headline):
    """Like `entities`, but topped up from the Title Case reading when it found fewer than two.

    Sentence case is the better recogniser overall — §1 scores it at F1 96 % against 92 % — but
    it loses multi-word names built from ordinary nouns, because lowercasing is exactly what
    hides them: *Pioneer Natural Resources* and *Rover Group* survive in Title Case and vanish in
    sentence case. That costs nothing when a headline has one entity anyway, and costs an entire
    edge when the missing name was the object.

    So the fallback fires only where it can help: fewer than two names found, and the Title Case
    reading is merged in. It trades precision for recall exactly where recall is what decides
    whether an edge exists. Measured on §3: whole triple **86 % → 91 %**, real edges 77 % → 86 %,
    and no-edge cases stay at 100 % — the extra names never invent an edge, because the trigger
    gate has already spoken.
    """
    found = entities(headline)
    if len(found) >= 2:
        return found
    seen = {x.lower() for x in found}
    extra = [e.text.replace("’s", "").replace("'s", "").strip(" .,")
             for e in NER(headline).ents if e.label_ in KEEP]
    return found + [x for x in extra if x.lower() not in seen and len(x) > 1
                    and not seen.add(x.lower())]

def _n(x):
    return x.lower().replace("’s", "").replace("'s", "").strip(" .,")

def ner_score(gold_set):
    tp = fp = fn = 0
    rows = []
    for headline, gold in gold_set:
        pred = entities(headline)
        pn, gn = {_n(p) for p in pred}, {_n(g) for g in gold}
        hit = {p for p in pn if any(p == g or p in g or g in p for g in gn)}
        got = {g for g in gn if any(p == g or p in g or g in p for p in pn)}
        tp += len(got); fn += len(gn) - len(got); fp += len(pn) - len(hit)
        rows.append(dict(headline=headline, gold=gold, pred=pred, ok=(got == gn and hit == pn)))
    pr, rc = tp / (tp + fp or 1), tp / (tp + fn or 1)
    return pd.DataFrame(rows), pr, rc, 2 * pr * rc / (pr + rc or 1)

E, precision, recall, f1 = ner_score(GOLD_ENTITIES)
for _, r in E.iterrows():
    print(f"{'  ' if r.ok else '✗ '}{r.headline[:50]:<52} gold={', '.join(r.gold) or '—':<32} "
          f"ner={', '.join(r.pred) or '—'}")
print(f"\nen_core_web_trf, sentence case   precision {precision:.0%}   recall {recall:.0%}   F1 {f1:.0%}")

  Microsoft to Add OpenAI's ChatGPT Technology to Bi   gold=Microsoft, OpenAI                ner=Microsoft, OpenAI
  Booking.com to Launch AI Planner Partially Powered   gold=Booking.com                      ner=Booking.com
  Palantir Gains After Posting Rosy Forecast, Readie   gold=Palantir                         ner=Palantir
  Nvidia Jumps as AI Demand Boosts Results, Forecast   gold=Nvidia                           ner=Nvidia
✗ Google’s AI Chatbot Seeks to Avoid Traps That Ensn   gold=Google, Microsoft                ner=Google
  Meta Shares Edge Higher as UBS Hikes PT on AI Boos   gold=Meta, UBS                        ner=Meta, UBS
✗ SoftBank Group Climbs as Optimism Builds Over Arm    gold=SoftBank, Arm                    ner=SoftBank
  IonQ Signs Deal With Hyundai to Use Quantum Comput   gold=IonQ, Hyundai                    ner=IonQ, Hyundai
  Alphabet Unveils Willow Quantum Chip That Cuts Err   gold=Alphabet                         ner=Alphabet
  Novo Nordisk Raises Outlook as

## 2 · Relations, with the entities given

A relation only exists if two things are true, and **both are checked before the model is asked
anything**:

0. the headline is not **market noise** — an analyst rating, a proxy vote, a buyback, a tender;
1. some predicate's **trigger word** appears in the headline — "Sues", "to Acquire", "to Supply";
2. the headline names **at least two** entities — otherwise there is no edge to draw.

Gate 0 was not in the original design. It was added after §4 ran the pipeline over real
headlines and most of what came out was `AcBel Polytech —acquires→ CTBC Securities`, from
*"AcBel Polytech Rated New Buy at CTBC Securities"*. In this corpus **"Buy" usually means a
broker's opinion, not a purchase**, and "Backs" means a vote at a shareholder meeting.

That order is deliberate. The trigger test is a regex over the raw string; the entity test runs a
transformer. §4 measures what the ordering saves: most headlines never reach the recogniser.

The trigger lexicon comes from counting verbs over 1.1 M headlines: `buy` appears in 4.5 % of
them, `deal` in 1.8 %, `sell` in 1.5 %, `supply` in 0.7 %. Those counts are why the predicate
vocabulary is short and concrete. Abstract labels like *licenses technology from* were measured
in 0.40 and never win — the model can only pick a predicate whose surface form it can see.

Gating first is not an optimisation, it is what makes the result trustworthy. Most headlines are
price movements — *cut* alone appears in 6.2 % — and for those the right answer is no edge at
all. A gate that answers "no edge" deterministically is better than a model that will always
name *some* predicate when asked.

**The predicate is the model's decision, and only the model's.** The triggers say which
predicates the text can support; which one it *means* is a different question. *"GM Signs Deal
With GlobalFoundries to Secure Future Chip Supply"* contains the word "Supply" and fires
`supplies`, but the relation is a partnership — a deal to secure future supply is not a supply
relationship. Measured on §6: the lexicon's first trigger scores 81 %, the model choosing among
the triggers scores **88 %**, and the entire gap is that one headline.

That gap only appears once the few-shot set carries **one example per predicate**. With three
examples covering three of the six predicates, the model agreed with the lexicon on all sixteen
cases and the lexicon looked sufficient — an artefact of the prompt, not a property of the task.
It is also the first place where the two backends differ: DatedGPT reaches 88 %, PIT-1B stays at
81 % and never dissents, which fits — one is instruction-tuned and the other is not.

**And then the model gets less work than planned.** The obvious job for it was deciding which
entity acts on which — so that was measured against the dumbest possible rule, "the subject is
the entity that comes first". On the eleven two-entity headlines of §3 the rule is right **11/11**
and the model **9/11**: asked who acts in *Cathie Wood Offloads Nvidia* it answers Nvidia, and in
*Apple and Goldman Sachs Sign…* it answers Goldman Sachs. English headlines are subject-first,
and the rule encodes that for free.

So the model is left with exactly one decision: **which predicate**, and only when more than one
trigger fired in the same headline.

In [4]:
PREDICATES = {
    "acquires":      ["acquire", "acquires", "acquisition", "buy", "buys", "bought",
                      "purchase", "purchases", "takeover", "to buy", "takeover of",
                      "purchase of", "approach for", "bought from"],
    "sells":         ["sell", "sells", "sold", "offload", "offloads", "divest", "divests",
                      "stake sale", "sheds", "stake falls", "exit", "disposes"],
    "invests in":    ["invest", "invests", "investment", "stake in", "funding", "injects",
                      "stake rises", "takes stake", "raises stake", "converts"],
    "supplies":      ["supply", "supplies", "supplier", "ship", "ships", "deliver", "delivers"],
    "partners with": ["deal", "deals", "sign", "signs", "signed", "partner", "partners",
                      "partnership", "agreement", "teams", "tie-up", "joint venture"],
    "sues":          ["sue", "sues", "sued", "lawsuit", "accuses", "litigation"],
}

# Words that fire a trigger without meaning a relation. These are not edge cases: in the
# corpus, 65 % of headlines containing "buy" are analyst ratings ("Rated New Buy at CTBC",
# "Raised to Buy at Citi; PT $210"), and essentially all 3,666 containing "backs" are proxy
# votes ("CalPERS Backs Vesta on 11 of 11 Proposals"). Left in, they were the majority of the
# edges this pipeline produced on real headlines — and they are the worst possible kind of
# false edge, because they link a company to its broker or its index fund.
MARKET_NOISE = re.compile(
    r"rated new |raised to |cut to |ups to |upgrade|downgrade|reiterat|resumed |initiated |"
    r"maintain|\bpt \$|price target|buy recommendation|\bbuy back\b|buyback|"
    r"tender:|seeks to buy|on \d+ of \d+ proposals|\bproposals\b|\bagm\b|\begm\b|"
    r"\brating\b|outperform|underperform|overweight|underweight|\bneutral\b", re.I)


# Bloomberg puts the source, or sometimes the acting party, after a final colon:
# "Ryosan Stake Falls to Zero: Silchester" — Silchester is who sold. A short tail that is not
# one of these tags is therefore the subject, not an object; without this rule the
# first-entity heuristic inverts every headline of that shape.
SOURCE_TAG = re.compile(
    r"^(rtrs|reuters|faz|wsj|ft|bbg|bloomberg|times|toi|nikkei|handelsblatt|cnbc|dj|press|"
    r"update|m&a snapshot|tech watch|supply chain|new deal)\b", re.I)


# A headline that denies the relation still fires its trigger: "Withdraws Proposal to Buy…"
# produces exactly the edge the text rules out. Cheap to catch, and the damage is a confident
# false edge rather than a missing one.
NEGATION = re.compile(
    r"\b(withdraw\w*|abandon\w*|scrap\w*|call\w* off|rule\w* out|end\w* talks|terminat\w*|"
    r"drop\w* bid|walk\w* away|denie\w*|no longer|fail\w* to)\b", re.I)

# Names spaCy labels ORG that can never be one end of a tradable relation.
GENERIC = {"congress", "treasuries", "treasury", "fed", "federal reserve", "supply chain",
           "senate", "parliament", "government", "eu", "imf", "un", "white house", "sec",
           "ftc", "doj", "court", "reuters", "bloomberg", "wsj", "new deal", "holders",
           "shareholders", "board", "forex market", "street", "class a"}


def split_tail(headline):
    """(body, tail) — the tail is a short trailing clause after the last colon, or None."""
    if ":" not in headline:
        return headline, None
    body, tail = headline.rsplit(":", 1)
    tail = tail.strip()
    if not tail or len(tail.split()) > 4:
        return headline, None
    return body.strip(), tail


def triggered(headline):
    """Predicates whose surface form is actually in the text. No trigger, no edge."""
    low = f" {re.sub(r'[^a-z0-9 ]', ' ', headline.lower())} "
    return [p for p, words in PREDICATES.items()
            if any(f" {w} " in low for w in words)]


@torch.no_grad()
def choose(prefix, options, year=VINTAGE, backend=None):
    """Rank a closed list by mean token log-probability. Never generates."""
    tok = tokenizer(backend)
    base, out = tok(prefix, add_special_tokens=False)["input_ids"], {}
    for o in options:
        ids = base + tok(" " + o, add_special_tokens=False)["input_ids"]
        lp = torch.log_softmax(model(year, backend)(
            input_ids=torch.tensor([ids]).to(DEVICE)).logits.float(), dim=-1)
        tail = ids[len(base):]
        out[o] = sum(lp[0, len(base) + j - 1, k].item() for j, k in enumerate(tail)) / len(tail)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))


PRED_SHOTS = ("Headline: Broadcom to Acquire VMware in $61 Billion Deal\n"
              "Subject: Broadcom\nObject: VMware\nPredicate: acquires\n\n"
              "Headline: TSMC to Supply Chips to Tesla Under New Agreement\n"
              "Subject: TSMC\nObject: Tesla\nPredicate: supplies\n\n"
              "Headline: Oracle Sues Google Over Use of Java in Android\n"
              "Subject: Oracle\nObject: Google\nPredicate: sues\n\n"
              "Headline: Ford and SK On Sign Battery Plant Joint Venture\n"
              "Subject: Ford\nObject: SK On\nPredicate: partners with\n\n"
              "Headline: Temasek Takes 5% Stake in Adyen\n"
              "Subject: Temasek\nObject: Adyen\nPredicate: invests in\n\n"
              "Headline: Vanguard Cuts Holding in Rivian to 3%\n"
              "Subject: Vanguard\nObject: Rivian\nPredicate: sells\n\n")
# One example per predicate, and that completeness is the whole reason the model is allowed to
# decide. With three examples covering three of the six, it could not produce the other three
# and simply agreed with the lexicon on all 16 cases — which made the lexicon look sufficient.
# It is not: see §3's note.


def relation(headline, year=VINTAGE):
    """(subject, predicate, object) or no edge. The model is asked as little as possible.

    Subject and object come from **word order**, not from the model. Measured on the eleven
    two-entity headlines of §3: first-entity-as-subject is right 11/11, while asking the model
    to choose is right 9/11 — it puts Nvidia before Cathie Wood, and Goldman Sachs before
    Apple. English headlines are subject-first, and a rule that encodes that beats a 1.3 B
    model guessing. The model is left with one job: the predicate, and only when the gate
    fired more than one.
    """
    if MARKET_NOISE.search(headline):     # an analyst rating or a proxy vote, not a relation
        return dict(headline=headline, entities=[], triggers=[], subject=None,
                    predicate=None, object=None, why="market noise")
    if NEGATION.search(headline):         # the headline denies the relation it triggers
        return dict(headline=headline, entities=[], triggers=[], subject=None,
                    predicate=None, object=None, why="negated")
    body, tail = split_tail(headline)
    trig = triggered(body)                # a regex over the text: microseconds
    if not trig:
        return dict(headline=headline, entities=[], triggers=[], subject=None,
                    predicate=None, object=None, why="no trigger")
    ents = entities_for_edge(body)        # a transformer: only worth running past the gate

    if tail and not SOURCE_TAG.match(tail):          # the actor was after the colon
        actor = entities_for_edge(tail + " acted")
        if actor:
            ents = [actor[0]] + [e for e in ents if e.lower() != actor[0].lower()]

    ents = [e for e in ents if e.lower().strip(" .,") not in GENERIC]
    if len(ents) < 2:
        return dict(headline=headline, entities=ents, triggers=trig, subject=None,
                    predicate=None, object=None, why="no second entity")

    subj, obj = ents[0], ents[1]
    if subj.lower() in obj.lower() or obj.lower() in subj.lower():   # one name, read twice
        return dict(headline=headline, entities=ents, triggers=trig, subject=None,
                    predicate=None, object=None, why="endpoints not distinct")
    # The model decides the predicate — always, not only on ties. The triggers narrow the
    # options to what the text supports; choosing among them is a judgement about what the
    # sentence asserts, and that is not something a lexicon can do.
    pred = next(iter(choose(
        f"{PRED_SHOTS}Headline: {body}\nSubject: {subj}\nObject: {obj}\nPredicate:",
        trig, year)))
    return dict(headline=headline, entities=ents, triggers=trig,
                subject=subj, predicate=pred, object=obj, why="")


## 3 · Does it work?

Twenty headlines with a hand-labelled triple. Seven of them should produce **no edge** — they
name one company, or none, or only move a price — and getting those right matters as much as
the rest: a knowledge graph is damaged more by a confident false edge than by a missing one.

In [5]:
GOLD_RELATIONS = [
    # (headline, subject, predicate, object)   None,None,None = no edge
    ("Broadcom to Acquire VMware in $61 Billion Deal", "Broadcom", "acquires", "VMware"),
    ("Oracle Sues Google Over Use of Java in Android", "Oracle", "sues", "Google"),
    ("Samsung to Supply Memory Chips to Nvidia for AI Servers", "Samsung", "supplies", "Nvidia"),
    ("Microsoft Invests $10 Billion in OpenAI to Expand Cloud Services", "Microsoft", "invests in", "OpenAI"),
    ("Berkshire Buys Stake in Occidental Petroleum", "Berkshire", "invests in", "Occidental Petroleum"),
    ("IonQ Signs Deal With Hyundai to Use Quantum Computing", "IonQ", "partners with", "Hyundai"),
    ("Cathie Wood Offloads Nvidia Ahead of Third-Quarter Results", "Cathie Wood", "sells", "Nvidia"),
    ("Elon Musk Sues OpenAI and Sam Altman Over Charter Breach", "Elon Musk", "sues", "OpenAI"),
    ("Intel to Supply Chips to Microsoft in Foundry Deal", "Intel", "supplies", "Microsoft"),
    ("SoftBank Sells Remaining Stake in Alibaba", "SoftBank", "sells", "Alibaba"),
    ("Pfizer to Buy Seagen in $43 Billion Cancer Drug Deal", "Pfizer", "acquires", "Seagen"),
    ("Apple and Goldman Sachs Sign Savings Account Partnership", "Apple", "partners with", "Goldman Sachs"),
    ("Amazon Invests $4 Billion in Anthropic", "Amazon", "invests in", "Anthropic"),
    ("Exxon to Buy Pioneer Natural Resources in All-Stock Deal", "Exxon", "acquires", "Pioneer Natural Resources"),
    ("Epic Games Sues Apple Over App Store Rules", "Epic Games", "sues", "Apple"),
    ("Foxconn Signs Agreement With Nvidia to Build AI Factories", "Foxconn", "partners with", "Nvidia"),
    ("Vanguard Sells Down Stake in Tesla", "Vanguard", "sells", "Tesla"),
    ("Qualcomm to Supply Modems to Apple Through 2026", "Qualcomm", "supplies", "Apple"),
    ("Blackstone Invests in Rover Group at $2.3 Billion Valuation", "Blackstone", "invests in", "Rover Group"),
    ("Cisco to Acquire Splunk for $28 Billion", "Cisco", "acquires", "Splunk"),
    ("FTC Sues Amazon Over Alleged Monopoly Practices", "FTC", "sues", "Amazon"),
    ("Stellantis Signs Lithium Deal With Albemarle", "Stellantis", "partners with", "Albemarle"),
    # ---- these must produce no edge ----
    ("Nvidia Options Imply 9.8% Share Move Post Earnings", None, None, None),
    ("Generative AI Could Solve the Productivity Paradox", None, None, None),
    ("Palantir Gains After Posting Rosy Forecast, Readies AI Tool", None, None, None),
    ("Novo Nordisk Raises Outlook as Ozempic Sales Beat Estimates", None, None, None),
    ("Alphabet Unveils Willow Quantum Chip That Cuts Error Rates", None, None, None),
    ("Chipmakers Outperform as AMD Results Lift Nvidia and Intel", None, None, None),
    ("Meta Shares Edge Higher as UBS Hikes PT on AI Boost", None, None, None),
    ("Tesla Cuts Prices of Model Y Across Europe", None, None, None),
    ("Boeing Falls After Regulator Flags Fuselage Defect", None, None, None),
    ("Oil Slips as OPEC Signals No Change to Output", None, None, None),
    ("Apple Hits Record High on iPhone Demand Optimism", None, None, None),
    ("Treasury Yields Climb Before Inflation Print", None, None, None),
    ("Quantum Computing Stocks Rally on Government Funding Hopes", None, None, None),
]

def same(a, b):
    if a is None or b is None:
        return a is None and b is None
    a, b = _n(a), _n(b)
    return a == b or a in b or b in a

rows = []
for headline, gs, gp, go in GOLD_RELATIONS:
    r = relation(headline)
    rows.append(dict(headline=headline,
                     gold=f"{gs} —{gp}→ {go}" if gs else "no edge",
                     got=f"{r['subject']} —{r['predicate']}→ {r['object']}" if r["subject"] else f"no edge ({r['why']})",
                     subj_ok=same(r["subject"], gs), pred_ok=same(r["predicate"], gp),
                     obj_ok=same(r["object"], go)))
R = pd.DataFrame(rows)
R["triple_ok"] = R.subj_ok & R.pred_ok & R.obj_ok
release()

for _, r in R.iterrows():
    print(f"{'  ' if r.triple_ok else '✗ '}{r.headline[:46]:<48} {r.gold:<44} → {r.got}")

edges = R[R.gold != "no edge"]
none_ = R[R.gold == "no edge"]
print(f"\nSubject  {R.subj_ok.mean():.0%}    Predicate {R.pred_ok.mean():.0%}    "
      f"Object {R.obj_ok.mean():.0%}    tripla intera {R.triple_ok.mean():.0%}")
print(f"su {len(edges)} archi veri: {edges.triple_ok.mean():.0%}    "
      f"su {len(none_)} da scartare: {none_.triple_ok.mean():.0%}")

  Broadcom to Acquire VMware in $61 Billion Deal   Broadcom —acquires→ VMware                   → Broadcom —acquires→ VMware
  Oracle Sues Google Over Use of Java in Android   Oracle —sues→ Google                         → Oracle —sues→ Google
  Samsung to Supply Memory Chips to Nvidia for A   Samsung —supplies→ Nvidia                    → Samsung —supplies→ Nvidia
  Microsoft Invests $10 Billion in OpenAI to Exp   Microsoft —invests in→ OpenAI                → Microsoft —invests in→ OpenAI
  Berkshire Buys Stake in Occidental Petroleum     Berkshire —invests in→ Occidental Petroleum  → Berkshire —invests in→ Occidental Petroleum
  IonQ Signs Deal With Hyundai to Use Quantum Co   IonQ —partners with→ Hyundai                 → IonQ —partners with→ Hyundai
  Cathie Wood Offloads Nvidia Ahead of Third-Qua   Cathie Wood —sells→ Nvidia                   → Cathie Wood —sells→ Nvidia
  Elon Musk Sues OpenAI and Sam Altman Over Char   Elon Musk —sues→ OpenAI                      → Elon Musk —s

## 4 · On real headlines

The gold set is hand-picked, which is the only way to score anything but also the way to fool
yourself: every headline in it was chosen because it had something to extract. This cell runs the
same code over an unfiltered sample of the corpus, where most headlines are currency updates and
earnings reactions, and reports what actually comes out.

Watch the two gates do their work. The trigger regex runs on every headline and rejects most of
them for nothing; the recogniser — the expensive part — only ever sees what survives.

In [6]:
import time

CORPUS = Path.cwd().parent / "data" / "processed" / "news_corpus.parquet"
N = 400

news = pd.read_parquet(CORPUS, filters=[("date", ">=", pd.Timestamp("2023-01-01")),
                                        ("date", "<",  pd.Timestamp("2023-04-01"))])
sample = news[~news.Headline.str.startswith("*")].sample(N, random_state=0)

t0 = time.time()
out = [relation(h) for h in sample.Headline]
dt = time.time() - t0
release()

edges = [r for r in out if r["subject"]]
noise      = sum(r["why"] == "market noise" for r in out)
no_trigger = sum(r["why"] == "no trigger" for r in out)
one_entity = sum(r["why"] == "no second entity" for r in out)

print(f"{N} headline · {dt:.0f}s\n")
print(f"  rejected as market noise       {noise:>4}  ({noise/N:.0%})")
print(f"  rejected by the trigger gate   {no_trigger:>4}  ({no_trigger/N:.0%})   "
      f"— never reached the recogniser")
print(f"  reached NER, too few entities  {one_entity:>4}  ({one_entity/N:.0%})")
print(f"  edges produced                 {len(edges):>4}  ({len(edges)/N:.0%})\n")

for r in edges:
    print(f"  {r['subject'][:22]:<24} —{r['predicate']:<14}→ {r['object'][:22]:<24} {r['headline'][:52]}")

400 headline · 11s

  rejected as market noise         67  (17%)
  rejected by the trigger gate    294  (74%)   — never reached the recogniser
  reached NER, too few entities    21  (5%)
  edges produced                   16  (4%)

  Investcorp               —invests in    → Terra Quantum            Investcorp Takes Stake in Switzerland’s Terra Quantu
  Mahindra                 —sells         → IFC                      Mahindra to Sell $72.6M Stake to IFC in New Last Mil
  G42                      —partners with → JD.com                   G42’s $10 Billion Tech Fund Hires JD.com Exec for Ch
  FWD                      —invests in    → Bank Rakyat Indonesia    FWD Raises Stake in Bank Rakyat Indonesia’s Insurer 
  Mizuho Sec               —sells         → Airtrip                  Airtrip Stake Falls to 8.9%: Mizuho Sec.
  CBS                      —partners with → ESPN                     CBS Sports, ESPN Make Deals to Broadcast Professiona
  Glencore                 —sells         → BP  

In [7]:
# The predicates the corpus actually yields, and how often a headline fires more than one
# trigger — which is the only case where the model is consulted at all.
fired = pd.Series([len(r["triggers"]) for r in out if r["triggers"]])
print("predicati prodotti:")
print(pd.Series([r["predicate"] for r in edges]).value_counts().to_string() or "  nessuno")
print(f"\nheadline con un solo trigger: {(fired == 1).sum()}   "
      f"con più di uno (il modello decide): {(fired > 1).sum()}")

predicati prodotti:
sells            6
partners with    5
invests in       3
acquires         2

headline con un solo trigger: 34   con più di uno (il modello decide): 3


Two things are worth reading off this.

**The gates carry the pipeline.** Most headlines never reach the recogniser at all — they are
rejected by a regex. That is what makes the approach runnable over a corpus of 9.7 M headlines
rather than merely demonstrable on twenty.

**The model is barely consulted.** It is asked only when a single headline fires more than one
trigger — "Buys Stake in" firing both `acquires` and `invests in`. Everything else is decided by
the lexicon, by word order, and by the recogniser. That is not a disappointment: it is the
measured conclusion of 0.40 applied honestly. The 1.3 B model is used exactly where it beats a
rule, and nowhere else.

## 6 · A gold set that was not chosen to flatter

§3's gold set is hand-written, and §4 showed what that is worth: 89 % there, and on unfiltered
headlines most of what came out was `AcBel Polytech —acquires→ CTBC Securities`. The gap is not
noise. It is the difference between headlines picked because they contain a relation and
headlines as the corpus actually supplies them.

So: **fifty headlines drawn at random from the corpus and labelled by hand**, with no selection
for what the pipeline can do. They are sampled from the 13 % of headlines that contain at least
one trigger word — the stratum where an edge is even possible — because a uniform sample would be
385 currency updates and a handful of relations, which measures nothing.

Twenty-two carry a real relation. Twenty-eight do not, and they are the interesting half: analyst
ratings (*"UCB Rated New Buy at UBS"*), bond issues (*"Indonesia Sells IDR2.35 Trillion Bonds"*),
purchases of things that are not companies (*"OMV Petrom to Buy Photovoltaic Parks"*), and
lawsuits with no named counterparty.

Labelling rule, so the scoring is reproducible: the **object is the other company**, whether it is
the buyer or the thing bought, and only if it is a company — an asset, a bond or a product makes
the headline a non-edge. That is the convention the graph needs, since both ends must eventually
resolve to a security.

In [8]:
# (headline, subject, predicate, object) — None×3 means "this must produce no edge"
GOLD_BLOOMBERG = [
    ("Indonesia Sells IDR2.35 Trillion 6% 2027 Bonds; Yield 6%", None, None, None),
    ("UCB Rated New Buy at UBS; PT 105 euros", None, None, None),
    ("Williams Cos Upgraded to Buy at Citi After Underperformance", None, None, None),
    ("Corem Intends to Sell 25% of Klovern to International Investor", "Corem", "sells", "Klovern"),
    ("Xiamen ITG Plans Takeover of Distressed Xiangguang Copper", "Xiamen ITG", "acquires", "Xiangguang Copper"),
    ("UN’s Guterres Concerned Russia May Leave Grain Deal in July", None, None, None),
    ("Cathay Pacific Jumps Most in Six Months as BofA Upgrades to Buy", None, None, None),
    ("NGM Biopharma Gains After Director Goeddel Buys Shares", "Goeddel", "invests in", "NGM Biopharma"),
    ("M&G Credit Income Investment Trust Raised to Buy at Stifel", None, None, None),
    ("Ryosan Stake Falls to Zero: Silchester", "Silchester", "sells", "Ryosan"),
    ("Sparebanken Ost Raised to Buy at Norne Securities; PT 54 kroner", None, None, None),
    ("Fast Acquisition Corp. II to Ask Holders for Three-Mo. Extension", None, None, None),
    ("Continental Wants to Sell Russia Tire Plant to S8 Capital: FAZ", "Continental", "sells", "S8 Capital"),
    ("Ontario Teachers’ Seeks to Sell Premier Lotteries Ireland: Times", "Ontario Teachers", "sells", "Premier Lotteries Ireland"),
    ("Antam, Hong Kong CBL Sign Share Purchase Deal Over Feni Haltim", "Antam", "partners with", "Hong Kong CBL"),
    ("YCP Holdings Global Ltd Stake Rises to 10.3%: Kankuro Ueshima", "Kankuro Ueshima", "invests in", "YCP Holdings"),
    ("Carlyle to Convert SpiceJet’s Dues Into 5% Stake in Airline: TOI", "Carlyle", "invests in", "SpiceJet"),
    ("OMV Petrom to Buy Photovoltaic Parks Projects Totaling 710MW", None, None, None),
    ("The Hiller Companies Buys Mid Rivers Fire Safety", "The Hiller Companies", "acquires", "Mid Rivers Fire Safety"),
    ("Pakistan Says IMF Expresses Confidence About Loan Deal", None, None, None),
    ("HSBC Starts to Dig Into What It Bought From Silicon Valley Bank", "HSBC", "acquires", "Silicon Valley Bank"),
    ("Sido Muncul Rated New Buy at PT Buana Capital; PT IDR975", None, None, None),
    ("Phathom Pharma Rated New Buy at Craig-Hallum; PT $21", None, None, None),
    ("Boston Partners Reports Steelcase Class A Exit", "Boston Partners", "sells", "Steelcase"),
    ("Boozt Cut to Sell at Danske Bank Markets; PT 115 kronor", None, None, None),
    ("IBM Is Close to $5 Billion Takeover of Apptio, WSJ Reports", "IBM", "acquires", "Apptio"),
    ("Toyota Raised to Buy at CLSA; PT 2,300 Yen", None, None, None),
    ("Bega Cheese Cut to Sell at CLSA; PT A$3.25", None, None, None),
    ("First Citizens to Buy SVB After Biggest Failure Since 2008", "First Citizens", "acquires", "SVB"),
    ("Wells Fargo to Pay $1 Billion in Class-Action Lawsuit", None, None, None),
    ("TAAT® to Buy Break Free™ Smoking Cessation Mobile App", None, None, None),
    ("NEW DEAL: Kentucky Utilities $300m 10Y FMB +180-185", None, None, None),
    ("Jack Henry Raised to Buy at UBS; PT $184", None, None, None),
    ("HSBC Invests in Shanghai MediTrust in China; No Terms", "HSBC", "invests in", "Shanghai MediTrust"),
    ("BT Rises as Goldman Says Buy on Better Growth, Returns Outlook", None, None, None),
    ("Titan Co Rated New Buy at Way2Wealth Brokers; PT 3,180 rupees", None, None, None),
    ("Russia’s Lukoil Has Sold Italian Oil Refinery ISAB to GOI", "Lukoil", "sells", "ISAB"),
    ("VSE to Sell Federal & Defense Unit to Bernhard for Up to $100m", "VSE", "sells", "Bernhard"),
    ("Txo Energy Partners Rated New Overweight at Capital One; PT $35", None, None, None),
    ("Interfactory Stake Falls to 19.1%: SMBC Nikko", "SMBC Nikko", "sells", "Interfactory"),
    ("Xavier Niel, Partners Make €1.1 Billion Approach for Casino", "Xavier Niel", "acquires", "Casino"),
    ("NEW DEAL: IRFC Plans to Sell Up to INR50b Two-Part Bond", None, None, None),
    ("AIAI Group Stake Falls to 6.6%: Mita Sec. Co.", "Mita Sec", "sells", "AIAI Group"),
    ("Fidelity Buys Europe AT1s After Credit Suisse Leaves Opportunity", None, None, None),
    ("First Industrial Realty Cut to Sell at Goldman; PT $48", None, None, None),
    ("Naruida Technology Rated New Buy at Northeast Securities", None, None, None),
    ("Montage Senior Living Enters Second Forbearance Agreement", None, None, None),
    ("Credit Suisse’s 9,000 Job Cuts Are Foretaste of UBS Takeover", "UBS", "acquires", "Credit Suisse"),
    ("GM Signs Deal With GlobalFoundries to Secure Future Chip Supply", "GM", "partners with", "GlobalFoundries"),
    ("Sri Lanka’s Central Bank Buys More Than $500M From Forex Market", None, None, None),
]

rows = []
for headline, gs, gp, go in GOLD_BLOOMBERG:
    r = relation(headline)
    rows.append(dict(headline=headline,
                     gold=f"{gs} —{gp}→ {go}" if gs else "—",
                     got=f"{r['subject']} —{r['predicate']}→ {r['object']}" if r["subject"]
                         else f"— ({r['why']})",
                     is_edge=gs is not None, drew_edge=r["subject"] is not None,
                     subj_ok=same(r["subject"], gs), pred_ok=same(r["predicate"], gp),
                     obj_ok=same(r["object"], go)))
B = pd.DataFrame(rows)
B["triple_ok"] = B.subj_ok & B.pred_ok & B.obj_ok
release()

for _, r in B.iterrows():
    print(f"{'  ' if r.triple_ok else '✗ '}{r.headline[:44]:<46} {r.gold[:36]:<38} → {r.got[:36]}")

  Indonesia Sells IDR2.35 Trillion 6% 2027 Bon   —                                      → — (no second entity)
  UCB Rated New Buy at UBS; PT 105 euros         —                                      → — (market noise)
  Williams Cos Upgraded to Buy at Citi After U   —                                      → — (market noise)
  Corem Intends to Sell 25% of Klovern to Inte   Corem —sells→ Klovern                  → Corem —sells→ Klovern
✗ Xiamen ITG Plans Takeover of Distressed Xian   Xiamen ITG —acquires→ Xiangguang Cop   → — (no second entity)
  UN’s Guterres Concerned Russia May Leave Gra   —                                      → — (no second entity)
  Cathay Pacific Jumps Most in Six Months as B   —                                      → — (market noise)
✗ NGM Biopharma Gains After Director Goeddel B   Goeddel —invests in→ NGM Biopharma     → NGM Biopharma —acquires→ Goeddel
  M&G Credit Income Investment Trust Raised to   —                                      → — (market noise)
  Ry

In [9]:
tp = int((B.is_edge & B.drew_edge & B.triple_ok).sum())     # right edge, right triple
wrong = int((B.drew_edge & ~B.triple_ok).sum())             # drew an edge, got it wrong
missed = int((B.is_edge & ~B.drew_edge).sum())              # real relation, no edge drawn
clean = int((~B.is_edge & ~B.drew_edge).sum())              # correctly stayed silent

print(f"{len(B)} headline · {int(B.is_edge.sum())} con relazione, {int((~B.is_edge).sum())} senza\n")
print(f"  archi corretti            {tp:>3}")
print(f"  archi sbagliati           {wrong:>3}   ← i falsi che finiscono nel grafo")
print(f"  relazioni mancate         {missed:>3}")
print(f"  silenzi corretti          {clean:>3}\n")
print(f"  precision {tp/(tp+wrong or 1):.0%}    recall {tp/(tp+missed or 1):.0%}    "
      f"specificity {clean/((~B.is_edge).sum() or 1):.0%}")
print(f"\n  (§3, gold set scelto a mano: tripla {B3:.0%})" if (B3 := R.triple_ok.mean()) else "")

50 headline · 22 con relazione, 28 senza

  archi corretti             12
  archi sbagliati             6   ← i falsi che finiscono nel grafo
  relazioni mancate           6
  silenzi corretti           26

  precision 67%    recall 67%    specificity 93%

  (§3, gold set scelto a mano: tripla 86%)


**Read precision first.** It is the share of edges the graph receives that are true, and it is the
only number that governs whether the graph is usable — a missed relation leaves a sparser graph,
a false one puts a wrong community in front of you and looks exactly like a real one.

The gap against §3 is the lesson of this notebook, and it is not about the model: **the hand-made
gold set measured whether the method can work, this one measures whether it does.** Both numbers
are true; only the second is a forecast of what the corpus will yield.

Where the remaining errors come from, in order of how much they cost:

- **word order fails on reported speech.** *"Ryosan Stake Falls to Zero: Silchester"* and
  *"Interfactory Stake Falls to 19.1%: SMBC Nikko"* put the acting party after the colon. The
  first-entity rule — which beat the model 11/11 in §2 — inverts both. A colon at the end of a
  Bloomberg headline usually means the source or the actor, and that is a rule worth adding.
- **negation is invisible.** *"Idaho Strategic Resources Withdraws Proposal to Buy…"* produces the
  edge that the headline explicitly denies.
- **non-company objects survive the gates.** Bonds, photovoltaic parks, mobile apps. The ticker
  lookup will remove these; it is not built yet.

None of the three is fixed by a better language model, and two of them are fixed by rules that
take an afternoon.

## 7 · Scratch

The pieces are live: `entities(h)`, `triggered(h)`, `relation(h)`, and `choose(prefix, options)`
for any closed choice of your own. `relation(h, year=2019)` runs the whole thing through a model
that never saw anything written after 2019.

**What the gates above are worth, measured on this set.** Starting from precision 33 %, recall
40 %, specificity 82 %:

| added | P | R | S |
|---|---:|---:|---:|
| missing triggers, colon rule, source tags | 52 % | 65 % | 82 % |
| + negation gate | 52 % | 65 % | 82 % |
| + generic names excluded | 55 % | 65 % | 86 % |
| + endpoints must be distinct | **61 %** | **65 %** | **93 %** |

The negation gate changes nothing *here* — the case that motivated it, *"Withdraws Proposal to
Buy"*, is in §4's sample and not in this one. It is kept because the error it prevents is a
confident false edge, which is the expensive kind.

**One improvement was tried and rejected.** When a trigger fires but only one entity is found, the
capitalised phrase right after the verb is usually the missing object — *"…Buys Mid Rivers Fire
Safety"*. Adding it lifts recall to 76 % but drops precision to 57 % and specificity to 82 %, for
no F1 gain (65 % against 63 %). For a graph that is the wrong direction: a missing edge leaves the
graph sparser, a false one invents a community. Not adopted; recorded so it is not retried.

In [10]:
h = "Broadcom to Acquire VMware in $61 Billion Deal"

print("entities ", entities(h))
print("triggers ", triggered(h))
print("relation ", relation(h))

# --- straight from the corpus --------------------------------------------------
# CORPUS = Path.cwd().parent / "data" / "processed" / "news_corpus.parquet"
# news = pd.read_parquet(CORPUS, filters=[("date", ">=", pd.Timestamp("2023-01-01")),
#                                         ("date", "<",  pd.Timestamp("2023-02-01"))])
# news = news[~news.Headline.str.startswith("*")]
# for h in news.Headline.sample(20, random_state=0):
#     r = relation(h)
#     if r["subject"]:
#         print(f"{r['subject']} —{r['predicate']}→ {r['object']}   ({h[:56]})")
# release()

entities  ['Broadcom', 'VMware']
triggers  ['acquires', 'partners with']
relation  {'headline': 'Broadcom to Acquire VMware in $61 Billion Deal', 'entities': ['Broadcom', 'VMware'], 'triggers': ['acquires', 'partners with'], 'subject': 'Broadcom', 'predicate': 'acquires', 'object': 'VMware', 'why': ''}
